# OCF v21 — pré-solve d+ FO econômica OCF

## Etapa A — pré-solve
Na primeira iteração resolve-se temporariamente:

\[
\min \sum_g I_{Gr,g}^2
\]

com as mesmas restrições KCL, Ohm e bounds.

O resultado \(x^{pre}\) é usado apenas como warm start.

## Etapa B — problema econômico

Em seguida resolve-se a FO econômica da v8:

\[
F_G=
\alpha\sum_g
\left[
a_gP_g^2+b_gP_g
\right]
\]

com a mesma escala numérica introduzida na v20.

Os lambdas e betas são obtidos **somente desta segunda solução econômica**.
Portanto o pré-solve não altera:

- o ótimo econômico;
- os multiplicadores econômicos;
- \(\beta_P,\beta_Q\);
- \(w\).

A regra de beta permanece:

1. primeira solução econômica com beta zero;
2. calcula \(\lambda\to\beta\to w\) uma única vez;
3. congela beta/w até a convergência.


In [ ]:

import os
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import osqp
from tabulate import tabulate

np.set_printoptions(precision=6, suppress=True)
print("Diretório atual:", os.getcwd())

from scipy.sparse.linalg import spsolve


Diretório atual: /content


## 1. Leitura dos dados

In [ ]:

def ler_dados_rede(
    caminho_barras="bus.csv",
    caminho_linhas="branch.csv",
    caminho_custos="g_costs.csv",
    caminho_bateria="store.csv",
    caminho_fo="fo.csv",
    caminho_curva_carga="load_flow.csv",
):
    dados_barra = pd.read_csv(caminho_barras)
    dados_linha = pd.read_csv(caminho_linhas)
    dados_custos = pd.read_csv(caminho_custos)

    dados_bateria = (
        pd.read_csv(caminho_bateria)
        if os.path.exists(caminho_bateria)
        else pd.DataFrame()
    )
    dados_fo = (
        pd.read_csv(caminho_fo)
        if os.path.exists(caminho_fo)
        else pd.DataFrame({"alfa": [1.0]})
    )
    dados_curva_carga = (
        pd.read_csv(caminho_curva_carga)
        if os.path.exists(caminho_curva_carga)
        else pd.DataFrame({"factorP": [1.0], "factorQ": [1.0]})
    )

    return (
        dados_barra,
        dados_linha,
        dados_custos,
        dados_bateria,
        dados_fo,
        dados_curva_carga,
    )


## 2. Estrutura do vetor de decisão

In [ ]:

def criar_indices_ocf(nb, nr, ngen):
    pos_Ir = 0
    pos_Ii = nr
    pos_Igr = 2 * nr
    pos_Igi = 2 * nr + ngen
    pos_Vr = 2 * nr + 2 * ngen
    pos_Vi = 2 * nr + 2 * ngen + nb
    nv = 2 * nr + 2 * ngen + 2 * nb

    return {
        "Ir": slice(pos_Ir, pos_Ir + nr),
        "Ii": slice(pos_Ii, pos_Ii + nr),
        "Igr": slice(pos_Igr, pos_Igr + ngen),
        "Igi": slice(pos_Igi, pos_Igi + ngen),
        "Vr": slice(pos_Vr, pos_Vr + nb),
        "Vi": slice(pos_Vi, pos_Vi + nb),
        "pos_Ir": pos_Ir,
        "pos_Ii": pos_Ii,
        "pos_Igr": pos_Igr,
        "pos_Igi": pos_Igi,
        "pos_Vr": pos_Vr,
        "pos_Vi": pos_Vi,
        "nv": nv,
    }


## 3. Matriz de igualdade: KCL + Lei de Ohm + Slack

In [ ]:

def criar_ysh_nodal(nb, de, para, bsht_linha=None, bsht_barra=None):
    """Monta a admitância shunt nodal usada na KCL.

    Convenção adotada:
      * bsht_barra é susceptância nodal, se fornecida;
      * bsht_linha é susceptância total do ramo e é dividida 50/50
        entre os dois extremos do ramo (modelo pi).

    Se os arquivos do caso não usam shunt, passe zeros/None.
    """
    ysh = np.zeros(nb, dtype=complex)

    if bsht_barra is not None:
        bbus = np.asarray(bsht_barra, dtype=float).ravel()
        if bbus.size == nb:
            ysh += 1j * bbus

    if bsht_linha is not None:
        bline = np.asarray(bsht_linha, dtype=float).ravel()
        de0 = np.asarray(de, dtype=int).ravel() - 1
        para0 = np.asarray(para, dtype=int).ravel() - 1
        if bline.size == de0.size:
            for ell in range(de0.size):
                ysh[de0[ell]] += 1j * bline[ell] / 2.0
                ysh[para0[ell]] += 1j * bline[ell] / 2.0

    return ysh


def criar_Aeq_ocf_ohm(
    nb,
    nr,
    ngen,
    de,
    para,
    r,
    xlin,
    gen_indices,
    slack_bus_num,
    ysh=None,
):
    """Cria Aeq para:

      [1] KCL real em nb barras
      [2] KCL imag em nb barras
      [3] Ohm real em nr ramos
      [4] Ohm imag em nr ramos
      [5] Vr_slack = Vf_r
      [6] Vi_slack = Vf_i

    KCL adotada:
        A_inc I_branch + I_G - Ysh V = I_D
    """
    de0 = np.asarray(de, dtype=int).ravel() - 1
    para0 = np.asarray(para, dtype=int).ravel() - 1
    r = np.asarray(r, dtype=float).ravel()
    xlin = np.asarray(xlin, dtype=float).ravel()
    gen_indices = np.asarray(gen_indices, dtype=int).ravel()

    if ysh is None:
        ysh = np.zeros(nb, dtype=complex)
    else:
        ysh = np.asarray(ysh, dtype=complex).ravel()

    idx = criar_indices_ocf(nb, nr, ngen)
    nv = idx["nv"]

    neq = 2 * nb + 2 * nr + 2
    Aeq = np.zeros((neq, nv), dtype=float)

    # --------------------------------------------------
    # KCL
    # --------------------------------------------------
    for ell in range(nr):
        k = de0[ell]
        m = para0[ell]

        # incidência: entrando +1 / saindo -1
        Aeq[k, idx["pos_Ir"] + ell] -= 1.0
        Aeq[m, idx["pos_Ir"] + ell] += 1.0

        Aeq[nb + k, idx["pos_Ii"] + ell] -= 1.0
        Aeq[nb + m, idx["pos_Ii"] + ell] += 1.0

    # geração nodal
    for g, bus in enumerate(gen_indices):
        Aeq[bus, idx["pos_Igr"] + g] += 1.0
        Aeq[nb + bus, idx["pos_Igi"] + g] += 1.0

    # -Ysh*V
    # YV = (G Vr - B Vi) + j(B Vr + G Vi)
    # -YV = (-G Vr + B Vi) + j(-B Vr - G Vi)
    for bus in range(nb):
        G = ysh[bus].real
        B = ysh[bus].imag

        Aeq[bus, idx["pos_Vr"] + bus] += -G
        Aeq[bus, idx["pos_Vi"] + bus] += B

        Aeq[nb + bus, idx["pos_Vr"] + bus] += -B
        Aeq[nb + bus, idx["pos_Vi"] + bus] += -G

    # --------------------------------------------------
    # Lei de Ohm por ramo
    # Vk - Vm - Zkm Ikm = 0
    # --------------------------------------------------
    off_or = 2 * nb
    off_oi = 2 * nb + nr

    for ell in range(nr):
        k = de0[ell]
        m = para0[ell]
        R = r[ell]
        X = xlin[ell]

        lr = off_or + ell
        li = off_oi + ell

        # real: Vk_r - Vm_r - R Ir + X Ii = 0
        Aeq[lr, idx["pos_Vr"] + k] += 1.0
        Aeq[lr, idx["pos_Vr"] + m] -= 1.0
        Aeq[lr, idx["pos_Ir"] + ell] -= R
        Aeq[lr, idx["pos_Ii"] + ell] += X

        # imag: Vk_i - Vm_i - X Ir - R Ii = 0
        Aeq[li, idx["pos_Vi"] + k] += 1.0
        Aeq[li, idx["pos_Vi"] + m] -= 1.0
        Aeq[li, idx["pos_Ir"] + ell] -= X
        Aeq[li, idx["pos_Ii"] + ell] -= R

    # --------------------------------------------------
    # Referência slack
    # --------------------------------------------------
    s = int(slack_bus_num) - 1
    Aeq[-2, idx["pos_Vr"] + s] = 1.0
    Aeq[-1, idx["pos_Vi"] + s] = 1.0

    return Aeq, idx


## 4. RHS da KCL: carga de potência constante

In [ ]:

def calcular_corrente_carga(P_pu, Q_pu, V):
    """I_D = conj(S_D / V), com S_D = P + jQ."""
    P_pu = np.asarray(P_pu, dtype=float).ravel()
    Q_pu = np.asarray(Q_pu, dtype=float).ravel()
    V = np.asarray(V, dtype=complex).ravel()

    Vseg = V.copy()
    mask = np.abs(Vseg) < 1e-8
    Vseg[mask] = 1.0 + 0.0j

    S = P_pu + 1j * Q_pu
    return np.conj(S / Vseg)


def montar_beq_ocf(Id, nb, nr, Vf_complex):
    """RHS compatível com criar_Aeq_ocf_ohm."""
    Id = np.asarray(Id, dtype=complex).ravel()
    beq = np.zeros(2 * nb + 2 * nr + 2, dtype=float)

    beq[0:nb] = Id.real
    beq[nb:2*nb] = Id.imag

    # Ohm = 0
    # Slack
    beq[-2] = float(np.real(Vf_complex))
    beq[-1] = float(np.imag(Vf_complex))
    return beq


## 5. Função objetivo com β marginal

A função objetivo do QP passa a conter duas parcelas:

$$F = F_G + F_{loss}.$$

Para o custo ativo dos geradores, mantém-se a aproximação diagonal da v7:

$$P_{G,g}^{(h)} \approx S_{base}V_{r,g}^{(h-1)}I_{G,g}^{r,(h)}.$$

Para as perdas dos ramos:

$$F_{loss}=\sum_{(k,m)} w_{km}\left[(I_{km}^r)^2+(I_{km}^i)^2\right],$$

onde:

$$w_{km}=R_{km}\beta_{P,km}+X_{km}\beta_{Q,km}.$$

Os betas econômicos são calculados a partir dos multiplicadores das linhas KCL.
A forma usada na Hessiana é escolhida separadamente para preservar a convexidade.


In [ ]:
def criar_H_f_ocf_corrente(
    nv, idx, nr, ngen,
    alpha,
    peso_beta_h,
):
    """
    FO OCF baseada diretamente em corrente:

        min alpha * sum_g(Igr_g^2)
            + sum_l peso_beta_h[l] * (Ir_l^2 + Ii_l^2)

    Como o OSQP usa:

        0.5*x.T*H*x + f.T*x

    então:

        H_Ir  = 2*peso_beta_h
        H_Ii  = 2*peso_beta_h
        H_Igr = 2*alpha

    Não há termo linear nesta versão.
    """
    H = np.zeros((nv, nv), dtype=float)
    f = np.zeros(nv, dtype=float)

    alpha = float(alpha)

    # Perdas dos ramos
    for ell in range(nr):
        w = float(peso_beta_h[ell])
        H[idx["pos_Ir"] + ell, idx["pos_Ir"] + ell] += 2.0 * w
        H[idx["pos_Ii"] + ell, idx["pos_Ii"] + ell] += 2.0 * w

    # Corrente ativa dos geradores
    for g in range(ngen):
        H[idx["pos_Igr"] + g, idx["pos_Igr"] + g] += 2.0 * alpha

    #imprimir_matriz_H(H)
    return H, f


## 6. Limites lineares

In [ ]:
def criar_bounds_ocf(
    dados_barra,
    nr,
    ngen,
    gen_indices,
    Sbase,
    Imax=100.0,
    vr_default=(0.80, 1.20),
    vi_default=(-0.50, 0.50),
):
    """Cria bounds de todas as variáveis.

    Observação: os limites Vr/Vi são uma aproximação retangular dos limites
    de tensão. Não representam exatamente Vmin <= |V| <= Vmax.
    """
    nb = len(dados_barra)
    idx = criar_indices_ocf(nb, nr, ngen)
    nv = idx["nv"]

    lb = -np.inf * np.ones(nv)
    ub =  np.inf * np.ones(nv)

    # corrente de ramo
    lb[idx["Ir"]] = -Imax
    ub[idx["Ir"]] =  Imax
    lb[idx["Ii"]] = -Imax
    ub[idx["Ii"]] =  Imax

    # geradores: mesma aproximação corrente ~ potência pu usada na base anterior
    Pmin = np.asarray(dados_barra["Pmin"], dtype=float) / Sbase
    Pmax = np.asarray(dados_barra["Pmax"], dtype=float) / Sbase
    Qmin = np.asarray(dados_barra["Qmin"], dtype=float) / Sbase
    Qmax = np.asarray(dados_barra["Qmax"], dtype=float) / Sbase

    for g, bus in enumerate(gen_indices):
        lb[idx["pos_Igr"] + g] = Pmin[bus]
        ub[idx["pos_Igr"] + g] = Pmax[bus]

        # Para V ≈ 1∠0:
        # Qg = Vi*Igr - Vr*Igi ≈ -Igi
        # portanto:
        #   Igi_min ≈ -Qmax
        #   Igi_max ≈ -Qmin
        lb[idx["pos_Igi"] + g] = -Qmax[bus]
        ub[idx["pos_Igi"] + g] = -Qmin[bus]

    # tensões: caixas retangulares
    Vmin = np.asarray(dados_barra.get("Vmin", pd.Series(np.zeros(nb))), dtype=float)
    Vmax = np.asarray(dados_barra.get("Vmax", pd.Series(np.zeros(nb))), dtype=float)

    for bus in range(nb):
        vrl = Vmin[bus] if Vmin[bus] > 0 else vr_default[0]
        vru = Vmax[bus] if Vmax[bus] > 0 else vr_default[1]

        lb[idx["pos_Vr"] + bus] = vrl
        ub[idx["pos_Vr"] + bus] = vru
        lb[idx["pos_Vi"] + bus] = vi_default[0]
        ub[idx["pos_Vi"] + bus] = vi_default[1]
    # A slack já está fixa em Aeq:
    #   Vr_slack = Re(Vf)
    #   Vi_slack = Im(Vf)
    # Portanto NÃO duplicamos essa igualdade nos bounds.
    lb[idx["pos_Vr"] + slack_idx] = -np.inf
    ub[idx["pos_Vr"] + slack_idx] =  np.inf
    lb[idx["pos_Vi"] + slack_idx] = -np.inf
    ub[idx["pos_Vi"] + slack_idx] =  np.inf


    return lb, ub


## 7. Diagnósticos elétricos

In [ ]:
def extrair_solucao_ocf(x_sol, idx):
    Ir = x_sol[idx["Ir"]]
    Ii = x_sol[idx["Ii"]]
    Igr = x_sol[idx["Igr"]]
    Igi = x_sol[idx["Igi"]]
    Vr = x_sol[idx["Vr"]]
    Vi = x_sol[idx["Vi"]]

    return {
        "Ibranch": Ir + 1j * Ii,
        "Ig": Igr + 1j * Igi,
        "V": Vr + 1j * Vi,
        "Ir": Ir,
        "Ii": Ii,
        "Igr": Igr,
        "Igi": Igi,
        "Vr": Vr,
        "Vi": Vi,
    }


def calcular_potencias_geradas_ocf(sol, gen_indices, Sbase_MVA):
    V = sol["V"]
    Ig = sol["Ig"]
    gen_indices = np.asarray(gen_indices, dtype=int)

    Sg = V[gen_indices] * np.conj(Ig) * Sbase_MVA
    return Sg.real, Sg.imag


def calcular_perdas_ramos_ocf(Ibranch, r, xlin, Sbase_MVA):
    I2 = np.abs(Ibranch)**2
    Ploss = np.asarray(r, dtype=float) * I2 * Sbase_MVA
    Qloss = np.asarray(xlin, dtype=float) * I2 * Sbase_MVA
    return Ploss, Qloss


def residuo_ohm(sol, de, para, r, xlin):
    V = sol["V"]
    I = sol["Ibranch"]
    de0 = np.asarray(de, dtype=int) - 1
    para0 = np.asarray(para, dtype=int) - 1
    Z = np.asarray(r, dtype=float) + 1j*np.asarray(xlin, dtype=float)
    return V[de0] - V[para0] - Z*I


## 8. Imprimir

In [ ]:
def imprimir_dados_entrada_ocf(
    dados_barra,
    dados_linha,
    dados_custos,
    num,
    tipo,
    gen_indices,
    lb,
    ub,
    idx,
    nr,
    ngen,
    Imax,
):
    """Imprime os dados de entrada em formato semelhante ao diagnóstico da v5."""

    print("\n" + "=" * 92)
    print("DADOS DAS BARRAS")
    print("=" * 92)

    cols_barra = [
        c for c in [
            "num", "tipo", "Pc", "Qc", "Pg", "Qg",
            "VM", "Th", "Vmin", "Vmax",
            "Pmin", "Pmax", "Qmin", "Qmax"
        ]
        if c in dados_barra.columns
    ]

    if cols_barra:
        print(
            tabulate(
                dados_barra[cols_barra].values.tolist(),
                headers=cols_barra,
                tablefmt="pretty",
                floatfmt=".4f",
            )
        )

    print("\n" + "=" * 92)
    print("DADOS DAS LINHAS")
    print("=" * 92)

    cols_linha = [
        c for c in ["de", "para", "r", "x", "bsht"]
        if c in dados_linha.columns
    ]

    tabela_linhas = []
    for ell in range(len(dados_linha)):
        tabela_linhas.append(
            [ell + 1] + [dados_linha.iloc[ell][c] for c in cols_linha]
        )

    print(
        tabulate(
            tabela_linhas,
            headers=["Ramo"] + cols_linha,
            tablefmt="pretty",
            floatfmt=".4f",
        )
    )

    print("\n" + "=" * 92)
    print("LIMITES DAS VARIÁVEIS")
    print("=" * 92)

    tabela_lim_gen = []
    for g, bus in enumerate(gen_indices):
        tabela_lim_gen.append([
            g + 1,
            int(num[bus]),
            lb[idx["pos_Igr"] + g],
            ub[idx["pos_Igr"] + g],
            lb[idx["pos_Igi"] + g],
            ub[idx["pos_Igi"] + g],
        ])

    if tabela_lim_gen:
        print("\nLimites dos geradores em corrente [pu]:")
        print(
            tabulate(
                tabela_lim_gen,
                headers=[
                    "Gen", "Barra",
                    "Igr min", "Igr max",
                    "Igi min", "Igi max"
                ],
                tablefmt="pretty",
                floatfmt=".4f",
            )
        )

    tabela_lim_v = []
    nb_local = len(num)
    for bus in range(nb_local):
        tabela_lim_v.append([
            int(num[bus]),
            lb[idx["pos_Vr"] + bus],
            ub[idx["pos_Vr"] + bus],
            lb[idx["pos_Vi"] + bus],
            ub[idx["pos_Vi"] + bus],
        ])

    print("\nLimites retangulares de tensão [pu]:")
    print(
        tabulate(
            tabela_lim_v,
            headers=["Barra", "Vr min", "Vr max", "Vi min", "Vi max"],
            tablefmt="pretty",
            floatfmt=".4f",
        )
    )

    print(f"\nLimite de corrente dos ramos: -{Imax:.6f} <= Ir,Ii <= {Imax:.6f} pu")

    print("\n" + "=" * 92)
    print("CUSTOS DOS GERADORES")
    print("=" * 92)

    tabela_custos = []
    for g in range(ngen):
        bus = gen_indices[g]

        a = float(dados_custos["a"].iloc[g]) if "a" in dados_custos.columns and g < len(dados_custos) else np.nan
        b = float(dados_custos["b"].iloc[g]) if "b" in dados_custos.columns and g < len(dados_custos) else np.nan
        c0 = float(dados_custos["c"].iloc[g]) if "c" in dados_custos.columns and g < len(dados_custos) else np.nan

        tabela_custos.append([
            g + 1,
            int(num[bus]),
            a, b, c0,
        ])

    if tabela_custos:
        print(
            tabulate(
                tabela_custos,
                headers=["Gen", "Barra", "a", "b", "c"],
                tablefmt="pretty",
                floatfmt=".4f",
            )
        )
    else:
        print("Nenhum custo de gerador encontrado.")

    print("\n" + "=" * 92)
    print("RESUMO DO MODELO")
    print("=" * 92)
    print(f"nb={len(num)} | nr={nr} | ngen={ngen} | nv={idx['nv']}")
    print(f"Aeq={Aeq.shape}")
    print(
        "Modelo atual: KCL + Lei de Ohm + tensões explícitas. "
        "Não são necessárias as matrizes LE/LI da formulação anterior."
    )


def calcular_lambda_pq_kcl(res, nb, V, sinal_lambda=1.0):
    """
    Extrai os duais das primeiras 2*nb linhas de Aeq:

      0:nb       -> KCL real
      nb:2*nb    -> KCL imaginária

    e converte lambda_r/lambda_i para lambda_P/lambda_Q.

    Relações:
        lambda_P = (Vr*lambda_r + Vi*lambda_i)/|V|^2
        lambda_Q = (Vi*lambda_r - Vr*lambda_i)/|V|^2
    """
    y = np.asarray(res.y, dtype=float).reshape(-1)

    if y.size < 2 * nb:
        raise ValueError(
            f"Vetor dual possui {y.size} elementos; esperados pelo menos {2*nb}."
        )

    lambda_r = sinal_lambda * y[0:nb]
    lambda_i = sinal_lambda * y[nb:2*nb]

    Vr = np.real(V)
    Vi = np.imag(V)
    V2 = Vr**2 + Vi**2
    V2 = np.maximum(V2, 1e-12)

    lambda_p = (Vr * lambda_r + Vi * lambda_i) / V2
    lambda_q = (Vi * lambda_r - Vr * lambda_i) / V2

    return {
        "lambda_r": lambda_r,
        "lambda_i": lambda_i,
        "lambda_p": lambda_p,
        "lambda_q": lambda_q,
    }


def calcular_beta_ramos(lambda_p, lambda_q, de, para):
    """
    beta_P_km = 0.5 * (lambda_P_k + lambda_P_m)
    beta_Q_km = 0.5 * (lambda_Q_k + lambda_Q_m)
    """
    lambda_p = np.asarray(lambda_p, dtype=float).ravel()
    lambda_q = np.asarray(lambda_q, dtype=float).ravel()

    de0 = np.asarray(de, dtype=int).ravel() - 1
    para0 = np.asarray(para, dtype=int).ravel() - 1

    beta_p = 0.5 * (lambda_p[de0] + lambda_p[para0])
    beta_q = 0.5 * (lambda_q[de0] + lambda_q[para0])

    return beta_p, beta_q

def converter_beta_para_hessiana(
    beta_p,
    beta_q,
    r,
    xlin,
    usar_beta_q=True,
    modo="abs_beta",
):
    """
    peso_economico = beta_P*R + beta_Q*X

    modos:
      economico  -> usa o peso com sinal; exige peso >= 0
      clip_zero  -> max(peso_economico, 0)
      abs_peso   -> abs(peso_economico)
      abs_beta   -> |beta_P|R + |beta_Q|X
    """
    beta_p = np.asarray(beta_p, dtype=float).ravel()
    beta_q = np.asarray(beta_q, dtype=float).ravel()
    r = np.asarray(r, dtype=float).ravel()
    xlin = np.asarray(xlin, dtype=float).ravel()

    if usar_beta_q:
        peso_economico = beta_p * r + beta_q * xlin
    else:
        peso_economico = beta_p * r

    if modo == "economico":
        peso_h = peso_economico.copy()
        if np.any(peso_h < -1e-12):
            raise ValueError(
                "Peso econômico negativo na Hessiana. "
                "Verifique SINAL_LAMBDA ou use outro MODO_BETA."
            )
    elif modo == "clip_zero":
        peso_h = np.maximum(peso_economico, 0.0)
    elif modo == "abs_peso":
        peso_h = np.abs(peso_economico)
    elif modo == "abs_beta":
        if usar_beta_q:
            peso_h = np.abs(beta_p) * r + np.abs(beta_q) * xlin
        else:
            peso_h = np.abs(beta_p) * r
    else:
        raise ValueError(
            "modo deve ser 'economico', 'clip_zero', 'abs_peso' ou 'abs_beta'."
        )

    return peso_economico, peso_h


In [ ]:
def criar_H_f_com_beta(
    nb,
    nr,
    ngen,
    alpha,
    a,
    b,
    V_prev,
    gen_indices,
    Sbase_MVA,
    peso_beta_h,
    usar_vr=True,
    escalar_fo=True,
    escala_min=1.0,
):
    """
    FO econômica:

        min 0.5*x.T*H*x + f.T*x

    Geração:
        Pg ~= Sbase_MVA * Vr_prev * IGr

        Cg(Pg) = alpha * (a_g*Pg^2 + b_g*Pg)

    Perdas:
        F_loss = sum_l peso_beta_h[l]*(Ir_l^2 + Ii_l^2)

    Escalonamento numérico:
        H_scaled = H / escala_fo
        f_scaled = f / escala_fo

    onde:
        escala_fo = max(abs(diag(H)), escala_min)

    O minimizador primal não é alterado. Os multiplicadores duais
    associados à FO original são recuperados por:

        lambda_original = lambda_scaled * escala_fo

    Retorna:
        H_scaled, f_scaled, escala_fo
    """
    idx = criar_indices_ocf(nb, nr, ngen)
    nv = idx["nv"]

    H = np.zeros((nv, nv), dtype=float)
    f = np.zeros(nv, dtype=float)

    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    V_prev = np.asarray(V_prev, dtype=complex).ravel()
    gen_indices = np.asarray(gen_indices, dtype=int).ravel()
    peso_beta_h = np.asarray(peso_beta_h, dtype=float).ravel()

    alpha = float(np.asarray(alpha).reshape(-1)[0])

    if a.size != ngen or b.size != ngen:
        raise ValueError(
            f"Custos incompatíveis: ngen={ngen}, len(a)={a.size}, len(b)={b.size}"
        )

    if peso_beta_h.size != nr:
        raise ValueError(
            f"peso_beta_h possui {peso_beta_h.size} posições, mas nr={nr}."
        )

    # -------------------------------------------------------------
    # PARCELA 1 DA FO: PERDAS
    #
    #   F_loss,l = w_l*(Ir_l^2 + Ii_l^2)
    #
    # Como o OSQP usa 0.5*x.T*H*x:
    #
    #   H[Ir_l,Ir_l] = 2*w_l
    #   H[Ii_l,Ii_l] = 2*w_l
    # -------------------------------------------------------------
    for ell in range(nr):
        H[idx["pos_Ir"] + ell, idx["pos_Ir"] + ell] += 2.0 * peso_beta_h[ell]
        H[idx["pos_Ii"] + ell, idx["pos_Ii"] + ell] += 2.0 * peso_beta_h[ell]

    # -------------------------------------------------------------
    # PARCELA 2 DA FO: CUSTO ECONÔMICO DA GERAÇÃO
    #
    #   Pg ~= Sbase_MVA * Vr_prev * IGr
    #
    #   ganho = Sbase_MVA * Vr_prev
    #
    #   Cg =
    #       alpha * [a_g*ganho^2*IGr^2 + b_g*ganho*IGr]
    #
    # Portanto:
    #
    #   H[IGr,IGr] += 2*alpha*a_g*ganho^2
    #   f[IGr]      +=   alpha*b_g*ganho
    # -------------------------------------------------------------
    for g, bus in enumerate(gen_indices):
        vr = float(V_prev[bus].real) if usar_vr else 1.0
        ganho = Sbase_MVA * vr
        pos = idx["pos_Igr"] + g

        H[pos, pos] += 2.0 * alpha * a[g] * ganho**2
        f[pos] += alpha * b[g] * ganho

    # -------------------------------------------------------------
    # ESCALONAMENTO GLOBAL DA FUNÇÃO OBJETIVO
    #
    # O termo (Sbase_MVA*Vr)^2 pode produzir elementos de H muito
    # maiores do que os termos de perdas. Para melhorar o
    # condicionamento numérico, dividimos TODA a FO pelo mesmo
    # escalar positivo.
    #
    #   H_scaled = H / escala_fo
    #   f_scaled = f / escala_fo
    #
    # O x ótimo é preservado.
    # -------------------------------------------------------------
    escala_fo = 1.0

    if escalar_fo:
        diag_H = np.abs(np.diag(H))
        diag_H_nz = diag_H[diag_H > 1e-12]

        if diag_H_nz.size > 0:
            escala_fo = max(
                float(np.max(diag_H_nz)),
                float(escala_min),
            )

            H = H / escala_fo
            f = f / escala_fo

    return H, f, escala_fo

In [ ]:
def imprimir_tensoes_finais(num, V):
    tabela_v = []

    for k, Vk in enumerate(V):
        tabela_v.append([
            int(num[k]),
            f"{np.real(Vk):.4f}",
            f"{np.imag(Vk):.4f}",
            f"{np.abs(Vk):.4f}",
            f"{np.angle(Vk, deg=True):.4f}",
        ])

    print("\n" + "=" * 92)
    print("TENSÕES FINAIS NAS BARRAS")
    print("=" * 92)

    print(
        tabulate(
            tabela_v,
            headers=[
                "Barra",
                "Vr [pu]",
                "Vi [pu]",
                "|V| [pu]",
                "Ângulo [graus]"
            ],
            tablefmt="pretty",
        )
    )

In [ ]:
def imprimir_matriz_H(H, titulo="Matriz H"):
    """
    Imprime a matriz H em formato de tabela com 3 casas decimais.

    Aceita:
      - numpy.ndarray
      - scipy.sparse matrix
    """

    # Converte sparse para array apenas para impressão
    if sp.issparse(H):
        H_print = H.toarray()
    else:
        H_print = np.asarray(H, dtype=float)

    nlin, ncol = H_print.shape

    # Nomes das colunas
    headers = [""] + [f"x{j+1}" for j in range(ncol)]

    # Monta tabela
    tabela = []

    for i in range(nlin):
        linha = [f"x{i+1}"]

        for j in range(ncol):
            linha.append(f"{H_print[i, j]:.3f}")

        tabela.append(linha)

    print(f"\n{titulo} ({nlin} x {ncol})")
    print(
        tabulate(
            tabela,
            headers=headers,
            tablefmt="pretty",
            stralign="right"
        )
    )

In [ ]:
# ==========================================================
# PARÂMETROS PRINCIPAIS
# ==========================================================
Sbase = 100.0
Vbase = 132.0
max_iter = 50
erro = 1e-4
Imax = 100.0

# ==========================================================
# PARÂMETROS DO BETA
# ==========================================================
USAR_BETA_Q = True
MODO_BETA = "abs_beta"  # economico | clip_zero | abs_peso | abs_beta
SINAL_LAMBDA = 1.0

# ==========================================================
# CONFIGURAÇÃO NUMÉRICA DO OSQP
# ==========================================================
OSQP_EPS_ABS = 1e-4
OSQP_EPS_REL = 1e-4
OSQP_MAX_ITER = 5000
USAR_WARM_START_DUAL = False

# ==========================================================
# LEITURA
# ==========================================================
(
    dados_barra,
    dados_linha,
    dados_custos,
    dados_bateria,
    dados_fo,
    dados_curva_carga,
) = ler_dados_rede()

num = np.asarray(dados_barra["num"], dtype=int)
tipo = np.asarray(dados_barra["tipo"], dtype=int)
VM = np.asarray(dados_barra["VM"], dtype=float)
Th = np.asarray(dados_barra["Th"], dtype=float)

Pc = np.asarray(dados_barra["Pc"], dtype=float) / Sbase
Qc = np.asarray(dados_barra["Qc"], dtype=float) / Sbase
Pg0 = np.asarray(dados_barra["Pg"], dtype=float) / Sbase
Qg0 = np.asarray(dados_barra["Qg"], dtype=float) / Sbase

# demanda líquida base
Pesp = Pc - Pg0
Qesp = Qc - Qg0

de = np.asarray(dados_linha["de"], dtype=int)
para = np.asarray(dados_linha["para"], dtype=int)
r = np.asarray(dados_linha["r"], dtype=float)
xlin = np.asarray(dados_linha["x"], dtype=float)
bsht_linha = (
    np.asarray(dados_linha["bsht"], dtype=float)
    if "bsht" in dados_linha.columns
    else np.zeros(len(dados_linha))
)

nb = len(dados_barra)
nr = len(dados_linha)
gen_indices = np.where((tipo == 2) | (tipo == 3))[0].tolist()
ngen = len(gen_indices)

slack_candidates = np.where(tipo == 3)[0]
if slack_candidates.size != 1:
    raise ValueError(f"Esperada exatamente uma barra slack; encontradas {slack_candidates.size}.")
slack_idx = int(slack_candidates[0])
slack_bus_num = int(num[slack_idx])

Vf_complex = VM[slack_idx] * np.exp(1j*np.deg2rad(Th[slack_idx]))

# custos de geração
bus_cost_a = np.asarray(dados_custos["a"], dtype=float)
bus_cost_b = np.asarray(dados_custos["b"], dtype=float)
bus_cost_c = np.asarray(dados_custos["c"], dtype=float)
alpha = (
    float(np.asarray(dados_fo["alfa"], dtype=float).reshape(-1)[0])
    if "alfa" in dados_fo.columns
    else 1.0
)

curva_cargaP = np.asarray(dados_curva_carga["factorP"], dtype=float)
curva_cargaQ = np.asarray(dados_curva_carga["factorQ"], dtype=float)

# shunt nodal (nesta primeira validação, use ZERO se quiser reproduzir
# exatamente o modelo série puro KCL + Ohm)
USAR_SHUNT_LINHA = False
if USAR_SHUNT_LINHA:
    ysh = criar_ysh_nodal(nb, de, para, bsht_linha=bsht_linha)
else:
    ysh = np.zeros(nb, dtype=complex)

Aeq, idx = criar_Aeq_ocf_ohm(
    nb=nb,
    nr=nr,
    ngen=ngen,
    de=de,
    para=para,
    r=r,
    xlin=xlin,
    gen_indices=gen_indices,
    slack_bus_num=slack_bus_num,
    ysh=ysh,
)

nv = idx["nv"]
lb, ub = criar_bounds_ocf(
    dados_barra=dados_barra,
    nr=nr,
    ngen=ngen,
    gen_indices=gen_indices,
    Sbase=Sbase,
    Imax=Imax,
)

# Slack também é igualdade em Aeq; deixamos os bounds coerentes.
lb[idx["pos_Vr"] + slack_idx] = Vf_complex.real
ub[idx["pos_Vr"] + slack_idx] = Vf_complex.real
lb[idx["pos_Vi"] + slack_idx] = Vf_complex.imag
ub[idx["pos_Vi"] + slack_idx] = Vf_complex.imag

print(f"nb={nb} | nr={nr} | ngen={ngen} | nv={nv}")
print(f"Aeq: {Aeq.shape}")
print(f"Slack: barra {slack_bus_num} | V={Vf_complex:.6f}")

imprimir_dados_entrada_ocf(
    dados_barra=dados_barra,
    dados_linha=dados_linha,
    dados_custos=dados_custos,
    num=num,
    tipo=tipo,
    gen_indices=gen_indices,
    lb=lb,
    ub=ub,
    idx=idx,
    nr=nr,
    ngen=ngen,
    Imax=Imax,
)

# ==========================================================
# FO OCF EM CORRENTE
# ==========================================================
ALPHA_CORRENTE = 1.0

# Na primeira iteração os betas são obrigatoriamente zero.
BETA_ZERO_NA_PRIMEIRA_ITERACAO = True
USAR_WARM_START_PRIMAL = True
FIXAR_BETA_ZERO = False

# ============================================================
# Escala numérica da função objetivo para o OSQP
# ============================================================
# Multiplicar/dividir toda a FO por uma constante positiva não altera
# a solução primal. A escala econômica dos duais é recuperada antes
# do cálculo de beta.
ESCALAR_FO_OSQP = True

# "auto": usa o maior valor absoluto da diagonal da Hessiana econômica
# da PRIMEIRA iteração como escala fixa para todo o período.
# É mantida fixa para que os duais possam ser corretamente reescalados.
ESCALA_FO_MODO = "auto"

# Limites de segurança para a escala automática.
ESCALA_FO_MIN = 1.0
ESCALA_FO_MAX = 1.0e8


# ============================================================
# v21 - pré-solução primal com a FO simples da v18
# ============================================================
# Usada SOMENTE para fornecer um bom warm start ao QP econômico.
# Não altera a função objetivo final nem os betas econômicos.
USAR_PRE_SOLVE = True
PRE_SOLVE_MAX_ITER = 5000


nb=69 | nr=68 | ngen=3 | nv=280
Aeq: (276, 280)
Slack: barra 1 | V=1.000000+0.000000j

DADOS DAS BARRAS
+------+------+-------+------+-----+-----+-----+-----+------+------+------+--------+---------+--------+
| num  | tipo |  Pc   |  Qc  | Pg  | Qg  | VM  | Th  | Vmin | Vmax | Pmin |  Pmax  |  Qmin   |  Qmax  |
+------+------+-------+------+-----+-----+-----+-----+------+------+------+--------+---------+--------+
| 1.0  | 3.0  |  0.0  | 0.0  | 0.0 | 0.0 | 1.0 | 0.0 | 0.0  | 0.0  | 0.0  | 2800.0 | -3000.0 | 3000.0 |
| 2.0  | 0.0  |  0.0  | 0.0  | 0.0 | 0.0 | 1.0 | 0.0 | 0.0  | 0.0  | 0.0  |  0.0   |   0.0   |  0.0   |
| 3.0  | 0.0  |  0.0  | 0.0  | 0.0 | 0.0 | 1.0 | 0.0 | 0.0  | 0.0  | 0.0  |  0.0   |   0.0   |  0.0   |
| 4.0  | 0.0  |  0.0  | 0.0  | 0.0 | 0.0 | 1.0 | 0.0 | 0.0  | 0.0  | 0.0  |  0.0   |   0.0   |  0.0   |
| 5.0  | 0.0  |  0.0  | 0.0  | 0.0 | 0.0 | 1.0 | 0.0 | 0.0  | 0.0  | 0.0  |  0.0   |   0.0   |  0.0   |
| 6.0  | 0.0  | 0.26  | 0.22 | 0.0 | 0.0 | 1.0 | 0.0 | 0.0  | 0.

## 9. Solução iterativa OCF/OSQP

## Ajuste numérico da FO — escala de \(H\) e \(f\)

Nesta versão, a formulação econômica não foi alterada.

A aproximação de potência ativa continua sendo

$$
P_g \approx S_{base}V_{r,g}^{prev}I_{Gr,g}.
$$

Logo, o termo quadrático do custo continua produzindo

$$
H_{gg}
=
2\alpha a_g
\left(S_{base}V_{r,g}^{prev}\right)^2,
$$

e o termo linear continua sendo

$$
f_g
=
\alpha b_g
\left(S_{base}V_{r,g}^{prev}\right).
$$

O ajuste realizado é exclusivamente numérico. Depois de montar a FO original, calcula-se

$$
s = \max_i |H_{ii}|,
$$

e o OSQP recebe

$$
\widetilde H=\frac{H}{s},
\qquad
\widetilde f=\frac{f}{s}.
$$

Como toda a função objetivo é dividida pelo mesmo escalar positivo,

$$
\arg\min F(x)
=
\arg\min \frac{F(x)}{s}.
$$

Portanto, o ponto ótimo primal é preservado.

Os multiplicadores duais, entretanto, precisam ser recuperados para a escala econômica original:

$$
\lambda_{\mathrm{econ}}
=
s\,\lambda_{\mathrm{scaled}}.
$$

Esse ajuste é aplicado antes da transformação

$$
(\lambda_r,\lambda_i)
\rightarrow
(\lambda_P,\lambda_Q)
\rightarrow
(\beta_P,\beta_Q).
$$


In [ ]:
def criar_solver_osqp(H, f, Aeq, beq, lb, ub, x_warm=None, y_warm=None):
    nv = H.shape[0]
    A_osqp = sp.vstack([
        sp.csc_matrix(Aeq),
        sp.eye(nv, format="csc"),
    ]).tocsc()

    l = np.concatenate([beq, lb])
    u = np.concatenate([beq, ub])

    solver = osqp.OSQP()
    solver.setup(
        P=sp.csc_matrix(H),
        q=np.asarray(f, dtype=float),
        A=A_osqp,
        l=np.asarray(l, dtype=float),
        u=np.asarray(u, dtype=float),
        verbose=False,
        polish=True,
        warm_starting=True,
        adaptive_rho=True,
        eps_abs=OSQP_EPS_ABS,
        eps_rel=OSQP_EPS_REL,
        max_iter=OSQP_MAX_ITER,
        check_termination=1,
    )

    if x_warm is not None:
        if y_warm is None:
            solver.warm_start(x=x_warm)
        else:
            solver.warm_start(x=x_warm)

    return solver



def criar_warm_start_fisico(Aeq, beq, lb, ub, x_fallback=None, tol_bounds=1e-8):
    """
    Para Aeq quadrada, resolve diretamente:

        Aeq x = beq

    e usa a solução como warm start somente se:
      - todos os valores forem finitos;
      - a solução respeitar os bounds.

    Para sistemas não quadrados, retorna x_fallback.
    """
    A_sp = sp.csc_matrix(Aeq)

    if A_sp.shape[0] != A_sp.shape[1]:
        return x_fallback, False

    try:
        x0 = spsolve(A_sp, np.asarray(beq, dtype=float))
    except Exception:
        return x_fallback, False

    if not np.all(np.isfinite(x0)):
        return x_fallback, False

    ok_lb = np.all(x0 >= np.asarray(lb) - tol_bounds)
    ok_ub = np.all(x0 <= np.asarray(ub) + tol_bounds)

    if not (ok_lb and ok_ub):
        return x_fallback, False

    residuo = np.max(np.abs(A_sp @ x0 - beq))
    if residuo > 1e-7:
        return x_fallback, False

    return np.asarray(x0, dtype=float), True



def criar_warm_start_dual_kkt(Aeq, H, f, x_primal):
    """
    Calcula lambda pela condição de estacionariedade:

        H x + f + Aeq.T lambda = 0

    Caso Aeq.T seja quadrada:
        resolve diretamente com spsolve.

    Caso Aeq.T seja retangular:
        calcula lambda por mínimos quadrados com LSQR.

    Retorna:
        lambda0
        residuo_dual_max
        metodo
    """

    Aeq = sp.csc_matrix(Aeq)
    Aeq_T = Aeq.T.tocsc()

    x_primal = np.asarray(x_primal, dtype=float).ravel()

    grad = np.asarray(
        H @ x_primal + f,
        dtype=float
    ).ravel()

    rhs = -grad

    # ============================================================
    # CASO 1 - sistema quadrado
    # ============================================================
    if Aeq_T.shape[0] == Aeq_T.shape[1]:

        try:
            lambda0 = spsolve(Aeq_T, rhs)
            metodo = "spsolve"

        except Exception as e:
            print(f"Falha no spsolve dual: {e}")
            return None, np.inf, "falha"

    # ============================================================
    # CASO 2 - sistema retangular
    #
    # Aeq.T lambda ~= -(Hx + f)
    #
    # mínimos quadrados
    # ============================================================
    else:

        resultado_lsqr = sp.linalg.lsqr(
            Aeq_T,
            rhs,
            atol=1e-10,
            btol=1e-10,
            iter_lim=10000,
        )

        lambda0 = resultado_lsqr[0]
        metodo = "lsqr"

    # ============================================================
    # Validação
    # ============================================================
    if lambda0 is None or not np.all(np.isfinite(lambda0)):
        return None, np.inf, metodo

    estacionariedade = (
        grad
        + np.asarray(Aeq_T @ lambda0).ravel()
    )

    residuo_dual = float(
        np.max(np.abs(estacionariedade))
    )

    print(
        f"Dual KKT: método={metodo} | "
        f"Aeq.T={Aeq_T.shape} | "
        f"resíduo={residuo_dual:.3e}"
    )

    return (
        np.asarray(lambda0, dtype=float),
        residuo_dual,
        metodo,
    )
    return np.asarray(lambda0, dtype=float), float(residuo_dual)


def atualizar_beta_apos_presolve(
    x_sol,
    idx,
    Aeq,
    nb,
    nr,
    ngen,
    alpha,
    a,
    b,
    gen_indices,
    Sbase_MVA,
    de,
    para,
    r,
    xlin,
    usar_beta_q=True,
    modo_beta="abs_beta",
    usar_vr=True,
):
    """
    Após o pré-solve:
      1) extrai V da solução primal;
      2) atualiza V_prev;
      3) monta a FO econômica com beta = 0;
      4) calcula lambda econômico via KKT:
             Aeq.T @ lambda = -(H @ x + f)
      5) transforma lambda de corrente para lambda_P/lambda_Q;
      6) calcula beta_P/beta_Q por ramo;
      7) converte beta em peso econômico e peso da Hessiana.

    Os duais do pré-solve NÃO são usados para beta.
    """
    sol_pre = extrair_solucao_ocf(x_sol, idx)
    V_prev = np.asarray(sol_pre["V"], dtype=complex).copy()

    peso_beta_zero = np.zeros(nr, dtype=float)

    H_econ, f_econ, escala_fo = criar_H_f_com_beta(
        nb=nb,
        nr=nr,
        ngen=ngen,
        alpha=alpha,
        a=a,
        b=b,
        V_prev=V_prev,
        gen_indices=gen_indices,
        Sbase_MVA=Sbase_MVA,
        peso_beta_h=peso_beta_zero,
        usar_vr=usar_vr,
    )
    lambda_eq, residuo_dual, metodo_dual = criar_warm_start_dual_kkt(
        Aeq=Aeq,
        H=H_econ,
        f=f_econ,
        x_primal=x_sol,
    )

    if lambda_eq is None:
        raise RuntimeError(
            "Não foi possível calcular os lambdas econômicos via KKT após o pré-solve."
        )

    # H_econ e f_econ foram divididos por escala_fo.
    # Portanto o lambda obtido pela KKT também está na escala da FO
    # escalada. Recuperamos o multiplicador da FO econômica original:
    #
    #     lambda_original = lambda_scaled * escala_fo
    lambda_eq = np.asarray(lambda_eq, dtype=float) * escala_fo

    lambda_r = np.asarray(lambda_eq[:nb], dtype=float)
    lambda_i = np.asarray(lambda_eq[nb:2*nb], dtype=float)

    Vr = V_prev.real
    Vi = V_prev.imag
    V2 = np.maximum(Vr**2 + Vi**2, 1e-12)

    lambda_p = (Vr * lambda_r + Vi * lambda_i) / V2
    lambda_q = (Vi * lambda_r - Vr * lambda_i) / V2

    beta_p, beta_q = calcular_beta_ramos(
        lambda_p=lambda_p,
        lambda_q=lambda_q,
        de=de,
        para=para,
    )

    peso_economico, peso_beta_h = converter_beta_para_hessiana(
        beta_p=beta_p,
        beta_q=beta_q,
        r=r,
        xlin=xlin,
        usar_beta_q=usar_beta_q,
        modo=modo_beta,
    )

    print("\n--- Beta calculado após pré-solve ---")
    print(f"Resíduo KKT dual = {residuo_dual:.3e}")
    print(f"Vmin pré-solve   = {np.min(np.abs(V_prev)):.6f} pu")
    print(f"Vmax pré-solve   = {np.max(np.abs(V_prev)):.6f} pu")
    print(f"lambdaP médio    = {np.mean(lambda_p):.6e}")
    print(f"lambdaQ médio    = {np.mean(lambda_q):.6e}")
    print(f"betaP médio      = {np.mean(beta_p):.6e}")
    print(f"betaQ médio      = {np.mean(beta_q):.6e}")
    print(f"peso H médio     = {np.mean(peso_beta_h):.6e}")

    return {
        "V_prev": V_prev,
        "H_econ": H_econ,
        "f_econ": f_econ,
        "lambda_eq": lambda_eq,
        "lambda_r": lambda_r,
        "lambda_i": lambda_i,
        "lambda_p": lambda_p,
        "lambda_q": lambda_q,
        "beta_p": beta_p,
        "beta_q": beta_q,
        "peso_economico": peso_economico,
        "peso_beta_h": peso_beta_h,
        "residuo_dual": residuo_dual,
        "escala_fo": escala_fo,
    }


historico_fos = []
historico_pg = []
historico_qg = []
historico_vmin = []
historico_perdas_p = []
historico_beta_p = []
historico_beta_q = []
historico_lambda_p = []
historico_lambda_q = []

V_inicial = np.ones(nb, dtype=complex)
V_inicial[slack_idx] = Vf_complex

x_warm_global = np.zeros(nv, dtype=float)
x_warm_global[idx["Vr"]] = V_inicial.real
x_warm_global[idx["Vi"]] = V_inicial.imag

for t in range(len(curva_cargaP)):
    ini_periodo = time.time()
    print(f"\n=========== Período {t+1}/{len(curva_cargaP)} ===========")

    P = Pesp * curva_cargaP[t]
    Q = Qesp * curva_cargaQ[t]

    V_prev = V_inicial.copy() if t == 0 else V_final.copy()

    x_sol = x_warm_global.copy()
    y_warm = None
    convergiu = False

    # primeira solução do período sem beta
    beta_p = np.zeros(nr)
    beta_q = np.zeros(nr)
    peso_economico = np.zeros(nr)
    peso_beta_h = np.zeros(nr)

    lambda_p = np.zeros(nb)
    lambda_q = np.zeros(nb)

    # v17: beta é calculado uma única vez, após a primeira solução.
    beta_calculado = False

    # v20: escala da FO é definida na primeira iteração e permanece
    # fixa durante todo o período.
    escala_fo_periodo = 1.0

    for h in range(max_iter):

        # 1) carga -> corrente
        Id = calcular_corrente_carga(P, Q, V_prev)
        beq = montar_beq_ocf(Id, nb, nr, Vf_complex)

        # ==========================================================
        # PRÉ-SOLUÇÃO PRIMAL COM A FO SIMPLES
        #
        # O objetivo desta etapa é somente encontrar um x praticamente
        # viável para as mesmas restrições Aeq/bounds. O resultado é usado
        # como warm start da FO econômica
        #
        # Esta etapa NÃO fornece lambda econômico e NÃO é usada para beta.
        # ==========================================================
        if h == 0 and USAR_PRE_SOLVE:
            H_pre, f_pre = criar_H_f_ocf_corrente(
                nv=nv,
                idx=idx,
                nr=nr,
                ngen=ngen,
                alpha=1.0,
                peso_beta_h=np.zeros(nr, dtype=float),
            )

            # pequena regularização numérica da pré-solução, como na v18
            H_pre = H_pre + np.eye(nv) * 1e-10

            solver_pre = criar_solver_osqp(
                H=H_pre,
                f=f_pre,
                Aeq=Aeq,
                beq=beq,
                lb=lb,
                ub=ub,
                x_warm=x_sol,
                y_warm=None,
            )

            res_pre = solver_pre.solve()

            print(
                f"Pré-solve v18: status={res_pre.info.status} | "
                f"iter={res_pre.info.iter} | "
                f"prim_res={res_pre.info.prim_res:.3e} | "
                f"dual_res={res_pre.info.dual_res:.3e}"
            )

            if res_pre.info.status_val in (1, 2):
                x_sol = res_pre.x.copy()
            elif res_pre.x is not None and np.all(np.isfinite(res_pre.x)):
                # Mesmo quando atinge o limite de iterações, aproveita-se o
                # melhor ponto primal disponível se ele for finito.
                x_sol = res_pre.x.copy()

            # NOVO: usa o x/V do pré-solve para obter o dual econômico via KKT
            # e, a partir dele, calcula beta e os pesos da Hessiana.
            resultado_beta_pre = atualizar_beta_apos_presolve(
                x_sol=x_sol,
                idx=idx,
                Aeq=Aeq,
                nb=nb,
                nr=nr,
                ngen=ngen,
                alpha=ALPHA_CORRENTE,
                a=bus_cost_a,
                b=bus_cost_b,
                gen_indices=gen_indices,
                Sbase_MVA=Sbase,
                de=de,
                para=para,
                r=r,
                xlin=xlin,
                usar_beta_q=USAR_BETA_Q,
                modo_beta=MODO_BETA,
                usar_vr=True,
            )

            V_prev = resultado_beta_pre["V_prev"]
            lambda_p = resultado_beta_pre["lambda_p"]
            lambda_q = resultado_beta_pre["lambda_q"]
            beta_p = resultado_beta_pre["beta_p"]
            beta_q = resultado_beta_pre["beta_q"]
            peso_economico = resultado_beta_pre["peso_economico"]
            peso_beta_h = resultado_beta_pre["peso_beta_h"]
            beta_calculado = True

        # 2) H da FO em corrente: beta=0 em h=0; beta congelado para h>=1

        # Regra da v11:
        # primeira iteração => beta = 0
        if h == 0 and BETA_ZERO_NA_PRIMEIRA_ITERACAO and not beta_calculado:
            peso_beta_h = np.zeros(nr, dtype=float)
            beta_p = np.zeros(nr, dtype=float)
            beta_q = np.zeros(nr, dtype=float)


        # v19: FO econômica da v8 sobre a estrutura escalável da v18
        # Pg é aproximado por:
        #     Pg ≈  Sbase* Vr_prev_g * IGr_g
        #
        # Na primeira iteração peso_beta_h = 0.
        # Da segunda em diante usamos o beta/w calculado uma única vez.
        Vr_gen_prev = np.asarray(
            [np.real(V_prev[bus]) for bus in gen_indices],
            dtype=float
        )

        H, f, escala_fo_periodo = criar_H_f_com_beta(
            nb=nb,
            nr=nr,
            ngen=ngen,
            alpha=ALPHA_CORRENTE,
            a=bus_cost_a,
            b=bus_cost_b,
            V_prev=V_prev,
            gen_indices=gen_indices,
            Sbase_MVA=Sbase,
            peso_beta_h=peso_beta_h,
            usar_vr=True,
        )

        print(
            f"Escala FO={escala_fo_periodo:.6e} | "
            f"max|diag(H_scaled)|={np.max(np.abs(np.diag(H))):.6e} | "
            f"max|f_scaled|={np.max(np.abs(f)):.6e}"
        )

        # Warm start dual via KKT
        lambda0 = None
        residuo_dual_ws = np.inf

        if USAR_WARM_START_DUAL and x_sol is not None:
            lambda0, residuo_dual_ws = criar_warm_start_dual_kkt(
                Aeq=Aeq,
                H=H,
                f=f,
                x_primal=x_sol,
            )

            residuo_primal_ws = np.max(
                np.abs(np.asarray(Aeq @ x_sol - beq).reshape(-1))
            )

            print(
                f"Warm start primal = {residuo_primal_ws:.3e} | "
                f"dual = {residuo_dual_ws:.3e}"
            )

        # 3) resolve QP
        residuo_ws_economico = float(
            np.max(np.abs(np.asarray(Aeq @ x_sol - beq).reshape(-1)))
        )

        solver = criar_solver_osqp(
            H=H,
            f=f,
            Aeq=Aeq,
            beq=beq,
            lb=lb,
            ub=ub,
            x_warm=x_sol,
            y_warm=y_warm,
        )

        res = solver.solve()

        print(
            f"OSQP status={res.info.status} | "
            f"iter={res.info.iter} | "
            f"prim_res={res.info.prim_res:.3e} | "
            f"dual_res={res.info.dual_res:.3e}"
        )

        if res.info.status_val not in (1, 2):
            raise RuntimeError(
                f"OSQP falhou no período {t+1}, iteração {h+1}: {res.info.status}"
            )

        x_sol = res.x.copy()
        y_warm = res.y.copy()

        sol = extrair_solucao_ocf(x_sol, idx)
        V_new = sol["V"]

        delta_complexo = float(np.max(np.abs(V_new - V_prev)))
        delta_modulo = float(np.max(np.abs(np.abs(V_new) - np.abs(V_prev))))

        Ploss_ramo, Qloss_ramo = calcular_perdas_ramos_ocf(
            sol["Ibranch"], r, xlin, Sbase
        )

        # ==================================================
        # 4) CALCULA lambda/beta APENAS APÓS A PRIMEIRA SOLUÇÃO
        #
        # h = 0:
        #   - a solução foi obtida com beta = 0
        #   - calculamos lambda e beta UMA ÚNICA VEZ
        #
        # h >= 1:
        #   - NÃO recalculamos lambda/beta
        #   - usamos os mesmos beta_p, beta_q e peso_beta_h
        #     até a convergência
        # ==================================================
        if h == 0 and not beta_calculado:

            # O OSQP resolveu F/escala_fo_periodo.
            # Portanto y_osqp = y_economico / escala_fo_periodo.
            # Recuperamos os duais econômicos antes de lambda -> beta.
            y_osqp_original = res.y.copy()
            res.y = res.y * escala_fo_periodo

            resultado_lambda = calcular_lambda_pq_kcl(
                res=res,
                nb=nb,
                V=V_new,
                sinal_lambda=SINAL_LAMBDA,
            )

            # restaura o vetor dual retornado originalmente pelo OSQP
            res.y = y_osqp_original

            lambda_p = np.asarray(
                resultado_lambda["lambda_p"], dtype=float
            )
            lambda_q = np.asarray(
                resultado_lambda["lambda_q"], dtype=float
            )

            # v18: cálculo de beta exatamente como na v8
            beta_p, beta_q = calcular_beta_ramos(
                lambda_p=lambda_p,
                lambda_q=lambda_q,
                de=de,
                para=para,
            )

            # v18: conversão beta -> w exatamente como na v8
            peso_economico, peso_beta_h = converter_beta_para_hessiana(
                beta_p=beta_p,
                beta_q=beta_q,
                r=r,
                xlin=xlin,
                usar_beta_q=USAR_BETA_Q,
                modo=MODO_BETA,
            )

            beta_calculado = True

            fo_economica = float(res.info.obj_val * escala_fo_periodo)

            print(
                f"Iter 0{h+1:02d} | "
                f"FO={fo_economica:.6f} | "
                f"ΔV={delta_complexo:.3e} | "
                f"Δ|V|={delta_modulo:.3e} | "
                f"Ploss={Ploss_ramo.sum():.6f} MW | "
                f"beta usado na FO=NÃO | "
                f"beta calculado agora=SIM | "
                f"betaP médio={np.mean(beta_p):.6e} | "
                f"betaQ médio={np.mean(beta_q):.6e} | "
                f"wH médio={np.mean(peso_beta_h):.6e} | "
                f"OSQP={res.info.iter}"
            )

        else:

            # A partir da segunda iteração os betas permanecem congelados.
            fo_economica = float(res.info.obj_val * escala_fo_periodo)

            print(
                f"Iter +{h+1:02d} | "
                f"FO={fo_economica:.6f} | "
                f"ΔV={delta_complexo:.3e} | "
                f"Δ|V|={delta_modulo:.3e} | "
                f"Ploss={Ploss_ramo.sum():.6f} MW | "
                f"beta usado na FO=SIM | "
                f"beta congelado=SIM | "
                f"betaP médio={np.mean(beta_p):.6e} | "
                f"betaQ médio={np.mean(beta_q):.6e} | "
                f"wH médio={np.mean(peso_beta_h):.6e} | "
                f"OSQP={res.info.iter}"
            )

        # Atualiza somente a tensão para a próxima iteração.
        # lambda, beta e pesos NÃO são recalculados após h=0.
        V_prev = V_new.copy()

        # Não encerra na primeira solução, pois ela foi feita com beta=0.
        # Exigimos ao menos uma solução posterior já com beta ativo na FO.
        if delta_complexo <= erro and h > 0:
            convergiu = True
            print(f"Tempo período = {time.time() - ini_periodo:.4f} s")
            break


    V_final = V_new.copy()
    x_warm_global = x_sol.copy()

    sol_final = extrair_solucao_ocf(x_sol, idx)

    Pg_MW, Qg_MVAr = calcular_potencias_geradas_ocf(
        sol_final, gen_indices, Sbase
    )

    Ploss_ramo, Qloss_ramo = calcular_perdas_ramos_ocf(
        sol_final["Ibranch"], r, xlin, Sbase
    )

    Pload_MW = float(np.sum(P) * Sbase)
    Qload_MVAr = float(np.sum(Q) * Sbase)

    balanco_P = float(np.sum(Pg_MW) - Pload_MW - np.sum(Ploss_ramo))
    balanco_Q = float(np.sum(Qg_MVAr) - Qload_MVAr - np.sum(Qloss_ramo))

    Id_final = calcular_corrente_carga(P, Q, V_final)
    beq_final = montar_beq_ocf(Id_final, nb, nr, Vf_complex)

    residuo_eq = float(np.max(np.abs(Aeq @ x_sol - beq_final)))

    print("\n--------------- RESULTADO FINAL ---------------")
    print(f"Convergiu: {convergiu}")
    print(f"Vmin = {np.min(np.abs(V_final)):.6f} pu")
    print(f"Vmax = {np.max(np.abs(V_final)):.6f} pu")
    print(f"ΣPg = {np.sum(Pg_MW):.6f} MW | ΣQg = {np.sum(Qg_MVAr):.6f} MVAr")
    print(f"ΣPload = {Pload_MW:.6f} MW | ΣQload = {Qload_MVAr:.6f} MVAr")
    print(f"Ploss = {np.sum(Ploss_ramo):.6f} MW | Qloss = {np.sum(Qloss_ramo):.6f} MVAr")
    print(f"Balanço P = {balanco_P:.6e} MW | Balanço Q = {balanco_Q:.6e} MVAr")
    print(f"max|Aeq*x-beq| = {residuo_eq:.3e}")

    imprimir_tensoes_finais(num, V_final)

    print(
        f"lambdaP médio = {np.mean(lambda_p):.6f} | "
        f"lambdaQ médio = {np.mean(lambda_q):.6f}"
    )

    print(
        f"betaP médio = {np.mean(beta_p):.6f} | "
        f"betaQ médio = {np.mean(beta_q):.6f}"
    )

    print(
        f"peso econômico médio = {np.mean(peso_economico):.6f} | "
        f"peso Hessiana médio = {np.mean(peso_beta_h):.6f}"
    )

    tabela_g = []
    for g, bus in enumerate(gen_indices):
        tabela_g.append([
            g + 1,
            int(num[bus]),
            Pg_MW[g],
            Qg_MVAr[g],
            np.abs(V_final[bus]),
            np.angle(V_final[bus], deg=True),
        ])

    # ==========================================================
    # TABELA DE GERAÇÃO
    # ==========================================================
    print(
        tabulate(
            [
                [
                    linha[0],              # Gen
                    linha[1],              # Barra
                    f"{linha[2]:.4f}",     # Pg
                    f"{linha[3]:.4f}",     # Qg
                    f"{linha[4]:.4f}",     # |V|
                    f"{linha[5]:.4f}",     # ângulo
                ]
                for linha in tabela_g
            ],
            headers=[
                "Gen",
                "Barra",
                "Pg [MW]",
                "Qg [MVAr]",
                "|V| [pu]",
                "ang V [graus]"
            ],
            tablefmt="pretty",
        )
    )


    # ==========================================================
    # TABELA DE LAMBDAS NODAIS
    # ==========================================================
    tabela_lambda = []

    for bus in range(nb):
        tabela_lambda.append([
            int(num[bus]),
            f"{lambda_p[bus]:.4f}",
            f"{lambda_q[bus]:.4f}",
        ])

    print("\nLambdas nodais:")

    print(
        tabulate(
            tabela_lambda,
            headers=[
                "Barra",
                "lambda_P",
                "lambda_Q"
            ],
            tablefmt="pretty",
        )
    )


    # ==========================================================
    # TABELA DE BETAS POR RAMO
    # ==========================================================
    tabela_beta = []

    for ell in range(nr):
        tabela_beta.append([
            ell + 1,
            int(de[ell]),
            int(para[ell]),
            f"{beta_p[ell]:.4f}",
            f"{beta_q[ell]:.4f}",
            f"{peso_economico[ell]:.4f}",
            f"{peso_beta_h[ell]:.4f}",
        ])

    print("\nBetas por ramo:")

    print(
        tabulate(
            tabela_beta,
            headers=[
                "Ramo",
                "De",
                "Para",
                "beta_P",
                "beta_Q",
                "w econômico",
                "w Hessiana"
            ],
            tablefmt="pretty",
        )
    )

    historico_fos.append(float(res.info.obj_val * escala_fo_periodo))
    historico_pg.append(Pg_MW.copy())
    historico_qg.append(Qg_MVAr.copy())
    historico_vmin.append(float(np.min(np.abs(V_final))))
    historico_perdas_p.append(float(np.sum(Ploss_ramo)))
    historico_beta_p.append(beta_p.copy())
    historico_beta_q.append(beta_q.copy())
    historico_lambda_p.append(lambda_p.copy())
    historico_lambda_q.append(lambda_q.copy())

USAR_WARM_START_PRIMAL = True
FIXAR_BETA_ZERO = False


=========== Período 1/1 ===========
Pré-solve v18: status=solved | iter=742 | prim_res=1.226e-05 | dual_res=2.707e-04
Dual KKT: método=lsqr | Aeq.T=(280, 276) | resíduo=7.251e-03

--- Beta calculado após pré-solve ---
Resíduo KKT dual = 7.251e-03
Vmin pré-solve   = 0.969183 pu
Vmax pré-solve   = 1.000000 pu
lambdaP médio    = -3.442362e+02
lambdaQ médio    = -1.995474e+01
betaP médio      = -3.443249e+02
betaQ médio      = -1.971569e+01
peso H médio     = 7.617206e-01
Escala FO=1.999999e+02 | max|diag(H_scaled)|=1.000000e+00 | max|f_scaled|=5.000002e-01
OSQP status=solved | iter=1486 | prim_res=2.101e-09 | dual_res=2.615e-11
Iter +01 | FO=929.424561 | ΔV=1.681e-01 | Δ|V|=1.177e-02 | Ploss=15.568772 MW | beta usado na FO=SIM | beta congelado=SIM | betaP médio=-3.443249e+02 | betaQ médio=-1.971569e+01 | wH médio=7.617206e-01 | OSQP=1486
Escala FO=2.000000e+02 | max|diag(H_scaled)|=1.000000e+00 | max|f_scaled|=5.000000e-01
OSQP status=solved | iter=1113 | prim_res=2.140e-09 | dual_res=2.